In [2]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

# =====================================================
# 1. LOAD DATA
# =====================================================
df = pd.read_excel(
    r"C:\Users\ahmed3farag\Downloads\RAW DATA-ev.xlsx"
)

# =====================================================
# 2. CLEAN COLUMN NAMES
# =====================================================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("COLUMNS:")
print(df.columns)

# =====================================================
# 3. DETECT IMPORTANT COLUMNS
# =====================================================
year_col = [c for c in df.columns if 'year' in c][0]

country_col = [c for c in df.columns if 'country' in c][0]

sales_col = None

possible_sales_cols = [
    'ev_sales',
    'sales',
    'total_sales',
    'units_sold'
]

for col in possible_sales_cols:
    if col in df.columns:
        sales_col = col
        break

# fallback
if sales_col is None:
    sales_col = df.columns[-1]

print("YEAR COLUMN:", year_col)
print("COUNTRY COLUMN:", country_col)
print("SALES COLUMN:", sales_col)

# =====================================================
# 4. CLEAN DATA TYPES
# =====================================================
df[year_col] = pd.to_numeric(
    df[year_col],
    errors='coerce'
)

df[sales_col] = pd.to_numeric(
    df[sales_col],
    errors='coerce'
)

df = df.dropna(
    subset=[year_col, sales_col]
)

# =====================================================
# 5. CREATE DASH APP
# =====================================================
app = dash.Dash(__name__)

# =====================================================
# 6. KPI CARD STYLE
# =====================================================
CARD_STYLE = {
    'padding': '25px',
    'backgroundColor': 'white',
    'borderRadius': '15px',
    'boxShadow': '0px 4px 10px rgba(0,0,0,0.1)',
    'textAlign': 'center',
    'flex': '1'
}

# =====================================================
# 7. APP LAYOUT
# =====================================================
app.layout = html.Div([

    # =================================================
    # TITLE
    # =================================================
    html.H1(
        "🚗 EV SALES DASHBOARD",
        style={
            'textAlign': 'center',
            'marginBottom': '30px',
            'color': '#2c3e50'
        }
    ),

    # =================================================
    # FILTERS
    # =================================================
    html.Div([

        dcc.Dropdown(
            id='country-filter',

            options=[
                {
                    'label': c,
                    'value': c
                }
                for c in sorted(
                    df[country_col]
                    .dropna()
                    .unique()
                )
            ],

            multi=True,

            placeholder="🌍 Select Country"

        ),

        html.Br(),

        dcc.RangeSlider(

            id='year-filter',

            min=int(df[year_col].min()),
            max=int(df[year_col].max()),

            value=[
                int(df[year_col].min()),
                int(df[year_col].max())
            ],

            marks={
                i: str(i)
                for i in range(
                    int(df[year_col].min()),
                    int(df[year_col].max()) + 1
                )
            }

        )

    ], style={'marginBottom': '30px'}),

    # =================================================
    # KPI ROW 1
    # =================================================
    html.Div([

        html.Div(id='kpi-total-sales', style=CARD_STYLE),

        html.Div(id='kpi-growth', style=CARD_STYLE),

        html.Div(id='kpi-top-country', style=CARD_STYLE)

    ],
    style={
        'display': 'flex',
        'gap': '20px',
        'marginBottom': '20px'
    }),

    # =================================================
    # KPI ROW 2
    # =================================================
    html.Div([

        html.Div(id='kpi-best-year', style=CARD_STYLE),

        html.Div(id='kpi-average-sales', style=CARD_STYLE),

        html.Div(id='kpi-total-countries', style=CARD_STYLE)

    ],
    style={
        'display': 'flex',
        'gap': '20px',
        'marginBottom': '20px'
    }),

    # =================================================
    # MAIN CHART
    # =================================================
    dcc.Graph(id='main-chart')

],
style={
    'padding': '30px',
    'backgroundColor': '#f5f7fa',
    'fontFamily': 'Arial'
})

# =====================================================
# 8. CALLBACK
# =====================================================
@app.callback(

    [
        Output('main-chart', 'figure'),

        Output('kpi-total-sales', 'children'),

        Output('kpi-growth', 'children'),

        Output('kpi-top-country', 'children'),

        Output('kpi-best-year', 'children'),

        Output('kpi-average-sales', 'children'),

        Output('kpi-total-countries', 'children')
    ],

    [
        Input('country-filter', 'value'),

        Input('year-filter', 'value')
    ]

)
def update_dashboard(countries, years):

    # =================================================
    # FILTER DATA
    # =================================================
    dff = df.copy()

    if countries:
        dff = dff[
            dff[country_col].isin(countries)
        ]

    dff = dff[
        (dff[year_col] >= years[0]) &
        (dff[year_col] <= years[1])
    ]

    # =================================================
    # EMPTY DATA CHECK
    # =================================================
    if dff.empty:

        empty_fig = px.line(
            title="No Data Available"
        )

        return (
            empty_fig,

            "No Data",
            "No Data",
            "No Data",
            "No Data",
            "No Data",
            "No Data"
        )

    # =================================================
    # KPI CALCULATIONS
    # =================================================

    # TOTAL SALES
    total_sales = dff[sales_col].sum()

    # AVERAGE SALES
    average_sales = dff[sales_col].mean()

    # TOTAL COUNTRIES
    total_countries = dff[country_col].nunique()

    # YEARLY SALES
    yearly_sales = (
        dff.groupby(year_col)[sales_col]
        .sum()
        .sort_index()
    )

    # GROWTH %
    if len(yearly_sales) > 1:

        previous_year = yearly_sales.iloc[-2]
        current_year = yearly_sales.iloc[-1]

        if previous_year != 0:

            growth = (
                (current_year - previous_year)
                / previous_year
            ) * 100

        else:
            growth = 0

    else:
        growth = 0

    # TOP COUNTRY
    top_country = (
        dff.groupby(country_col)[sales_col]
        .sum()
        .idxmax()
    )

    # BEST YEAR
    best_year = yearly_sales.idxmax()

    # =================================================
    # COLORS
    # =================================================
    growth_color = (
        'green'
        if growth >= 0
        else 'red'
    )

    # =================================================
    # MAIN CHART
    # =================================================
    fig = px.line(

        dff,

        x=year_col,

        y=sales_col,

        color=country_col,

        markers=True,

        title="EV Sales Over Time"

    )

    fig.update_layout(

        template='plotly_white',

        hovermode='x unified',

        title_font_size=24,

        xaxis_title='Year',

        yaxis_title='Sales'

    )

    # =================================================
    # KPI COMPONENTS
    # =================================================

    total_sales_kpi = html.Div([

        html.H3("🚗 Total Sales"),

        html.H1(
            f"{int(total_sales):,}"
        )

    ])

    growth_kpi = html.Div([

        html.H3("📈 Growth Rate"),

        html.H1(

            f"{growth:.2f}%",

            style={
                'color': growth_color
            }

        )

    ])

    top_country_kpi = html.Div([

        html.H3("🌍 Top Country"),

        html.H1(top_country)

    ])

    best_year_kpi = html.Div([

        html.H3("🏆 Best Year"),

        html.H1(str(best_year))

    ])

    average_sales_kpi = html.Div([

        html.H3("💰 Average Sales"),

        html.H1(
            f"{average_sales:,.0f}"
        )

    ])

    total_countries_kpi = html.Div([

        html.H3("🌎 Countries"),

        html.H1(str(total_countries))

    ])

    # =================================================
    # RETURN
    # =================================================
    return (

        fig,

        total_sales_kpi,

        growth_kpi,

        top_country_kpi,

        best_year_kpi,

        average_sales_kpi,

        total_countries_kpi

    )

# =====================================================
# 9. RUN APP
# =====================================================
app.run(
    debug=False,
    port=8052
)

COLUMNS:
Index(['country', 'region', 'year', 'vehicle_segment', 'powertrain_type',
       'ev_sales', 'petrol_car_sales', 'diesel_car_sales',
       'total_vehicle_sales', 'ev_market_share', 'charging_stations',
       'fast_chargers_share', 'avg_ev_range_km', 'fuel_price_usd_per_liter',
       'electricity_price_usd_per_kwh', 'gdp_per_capita',
       'urban_population_percent', 'co2_emissions_transport_mt',
       'ev_subsidy_usd', 'emission_regulation_score', 'ev_growth_rate_yoy',
       'is_ev_dominant'],
      dtype='object')
YEAR COLUMN: year
COUNTRY COLUMN: country
SALES COLUMN: ev_sales
